# RNNs with Affon
This notebook focuses on the built-in recurrent layers in `affon:nn`. It shows the difference between `nn.SimpleRNN(...)` for a single sequence, `nn.RNN(...)` for batched Elman recurrence, and `nn.LSTM(...)` for gated recurrent state.


In [1]:
import { adam, clear_grad, grad, tensor } from 'affon:compute'
import nn from 'affon:nn'



## Single Sequence: `nn.SimpleRNN`
A `SimpleRNN` consumes one sequence shaped `[seq_len, input_size]` and internally loops over token positions. The final hidden state across stacked layers is available through `hidden_state()`.


In [2]:
const simple = nn.SimpleRNN(3, 4, { num_layers: 2 })

const xSeq = tensor([
  [0.1, 0.2, 0.3],
  [0.3, 0.2, 0.1],
  [0.5, 0.4, 0.3],
  [0.7, 0.8, 0.9],
])

const ySeq = simple(xSeq)
console.log('SimpleRNN output shape:', ySeq.shape)
console.log('SimpleRNN hidden shape:', simple.hidden_state()?.shape)
console.log('SimpleRNN parameter names:', simple.parameters.named())


SimpleRNN output shape: [ 4, 4 ]
SimpleRNN hidden shape: [ 2, 4 ]
SimpleRNN parameter names: [ [ layers.0.weight_ih, [3×4 Tensor, dtype=f32]
             0          1          2          3
  ┌                                              ┐
0 │  -0.645566   -0.70251    -0.5608   -1.38135  │
1 │  -0.296263  -1.331159   0.585678   0.837331  │
2 │  -0.652188  -1.145876  -0.344719  -1.179011  │
  └                                              ┘
 ], [ layers.0.weight_hh, [4×4 Tensor, dtype=f32]
             0          1          2          3
  ┌                                              ┐
0 │  -1.045279  -0.924655  -1.018947  -0.513868  │
1 │   1.186646  -1.014706  -0.354204  -0.033323  │
2 │  -0.354105  -0.812487  -1.113352  -0.040636  │
3 │  -0.458223   -0.08944  -0.695555   1.190462  │
  └                                              ┘
 ], [ layers.0.bias, [1×4 Tensor, dtype=f32]
     0  1  2  3
  ┌              ┐
0 │  0  0  0  0  │
  └              ┘
 ], [ layers.1.weight_ih, [4×4 Ten

## Batched Sequence-First Input: `nn.RNN`
`nn.RNN(...)` is the classic sequence-first API. Inputs are shaped `[seq_len, batch, input_size]`, outputs are `[seq_len, batch, hidden_size]`, and `hidden_state()` returns `[num_layers, batch, hidden_size]`.


In [3]:
const rnn = nn.RNN(3, 4, { num_layers: 2 })

const xBatch = tensor([
  [
    [0.1, 0.2, 0.3],
    [0.3, 0.2, 0.1],
  ],
  [
    [0.5, 0.4, 0.3],
    [0.2, 0.4, 0.6],
  ],
  [
    [0.7, 0.8, 0.9],
    [0.9, 0.8, 0.7],
  ],
])

const yBatch = rnn(xBatch)
console.log('RNN output shape:', yBatch.shape)
console.log('RNN hidden shape:', rnn.hidden_state()?.shape)


RNN output shape: [ 3, 2, 4 ]
RNN hidden shape: [ 2, 2, 4 ]


## Explicit Initial Hidden State
For stacked batched RNNs, the initial hidden state has shape `[num_layers, batch, hidden_size]`.


In [4]:
const h0 = tensor([
  [
    [0, 0, 0, 0],
    [0, 0, 0, 0],
  ],
  [
    [0, 0, 0, 0],
    [0, 0, 0, 0],
  ],
])

const yBatch2 = rnn(xBatch, h0)
console.log('RNN output shape with explicit h0:', yBatch2.shape)


RNN output shape with explicit h0: [ 3, 2, 4 ]


## Batch-First Input
If your data pipeline is already batch-first, `nn.RNN(..., { batch_first: true })` expects `[batch, seq_len, input_size]` and returns `[batch, seq_len, hidden_size]`.


In [5]:
const rnnBatchFirst = nn.RNN(3, 4, { num_layers: 2, batch_first: true })

const xBatchFirst = tensor([
  [
    [0.1, 0.2, 0.3],
    [0.5, 0.4, 0.3],
    [0.7, 0.8, 0.9],
  ],
  [
    [0.3, 0.2, 0.1],
    [0.2, 0.4, 0.6],
    [0.9, 0.8, 0.7],
  ],
])

const yBatchFirst = rnnBatchFirst(xBatchFirst)
console.log('RNN batch_first output shape:', yBatchFirst.shape)
console.log('RNN batch_first hidden shape:', rnnBatchFirst.hidden_state()?.shape)


RNN batch_first output shape: [ 2, 3, 4 ]
RNN batch_first hidden shape: [ 2, 2, 4 ]


## LSTM
`nn.LSTM(...)` keeps both a hidden state and a cell state. It follows the same sequence-first default as `nn.RNN(...)`, and also supports `batch_first: true`.


In [6]:
const lstm = nn.LSTM(3, 4, { num_layers: 2, batch_first: true })

const yLstm = lstm(xBatchFirst)
console.log('LSTM output shape:', yLstm.shape)
console.log('LSTM hidden shape:', lstm.hidden_state()?.shape)
console.log('LSTM cell shape:', lstm.cell_state()?.shape)


LSTM output shape: [ 2, 3, 4 ]
LSTM hidden shape: [ 2, 2, 4 ]
LSTM cell shape: [ 2, 2, 4 ]


## Tiny Training Step
The recurrent forward examples above run correctly. This final cell also uses the public parameter-centered training path: `clear_grad(params)`, `grad(loss, params)`, and `step(params)`.


In [7]:
const model = nn.Sequential(
  nn.RNN(3, 4),
  x => x,
)

const criterion = nn.MSELoss()
const params = model.parameters
const step = adam({ lr: 1e-2 })

const target = tensor([
  [
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
  ],
  [
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
  ],
  [
    [0.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0],
  ],
])

clear_grad(params)
const pred = model(xBatch)
const loss = criterion(pred, target)

console.log('pred shape:', pred.shape)
console.log('loss:', loss.item())
grad(loss, params)
step(params)
console.log('optimizer ready:', step !== null)


pred shape: [ 3, 2, 4 ]
loss: 0.27089807391166687
optimizer ready: true
